In [1]:
import os
from pathlib import Path
print(os.getcwd())
ROOT = Path(os.environ.get("HOME_PROJ_DIR", Path.cwd().resolve().parents[1]))
os.chdir(ROOT)
print(os.getcwd())

/home/fatemeh/thesis/kinodata-3D-affinity-prediction/prob/probing_notebooks
/home/fatemeh/thesis/kinodata-3D-affinity-prediction


In [2]:
from kinodata.data import KinodataDocked
from kinodata.transform import TransformToComplexGraph

In [3]:
dataset = KinodataDocked(transform=TransformToComplexGraph(remove_heterogeneous_representation=False), use_multiprocessing=True, num_processes= 16)
dataset

: 

In [4]:
dataset[0]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVEFCKFGNLSTYLRSFLASRKCIHRDLAARNILLICDFGLA',
  scaffold='C1CCC(CC2CCCC(C3CC(C4CCCC4)C4CCCCC34)C2)CC1',
  activity_type='pIC50',
  ident=[1],
  smiles='Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1',
  ligand={
    z=[28],
    x=[28, 12],
    pos=[28, 3],
  },
  pocket={
    z=[652],
    x=[652, 12],
    pos=[652, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[680, 12],
    z=[680],
    pos=[680, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 64],
    edge_attr=[64, 4],
  },
  (pocket, bond, pocket)={
    edge_index=[2, 1308],
    edge_attr=[1308, 4],
  },
  (complex, bond, complex)={
    edge_index=[2, 1372],
    edge_attr=[1372, 4],
  }
)

# Calculate #N

In [ ]:
from prob.prob_targets import count_n_dataset
from prob.paths_and_io import save_out_tensor

nitrogen_atoms = count_n_dataset(dataset)
save_out_tensor(nitrogen_atoms, output_dir =  ROOT/"data/probing/targets", filename = "nitrogen_counts.pt")

100%|██████████| 119522/119522 [01:38<00:00, 1213.96it/s]


# Finding the anomaly

In [6]:
from prob.paths_and_io import load_out_tensor

loaded_nitrogen_atoms = load_out_tensor(output_dir = ROOT/"data/probing/targets", filename = "nitrogen_counts.pt")


In [7]:
max_ident, max_n = max(loaded_nitrogen_atoms.items(), key=lambda x: x[1])

In [8]:
idents = dataset.data.ident.tolist() 
max_idx = idents.index(max_ident)

/opt/conda/lib/python3.10/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The data of the dataset is already cached, so any modifications to `data` will not be reflected when accessing its elements. Clearing the cache now by removing all elements in `dataset._data_list`. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [ ]:
dataset[max_idx]

HeteroData(
  y=[1],
  docking_score=[1],
  posit_prob=[1],
  predicted_rmsd=[1],
  pocket_sequence='VKLGQGCFGEVWMVAIKTLAFLQEAQVMKKLREKLVQLYAVYIVGEYMSKGSLLDFLKGYVERMNYVHRDLRAANILVVADFGLA',
  scaffold='CC(CCCC1CCC2CCCCC21)CCC(C)C(CC(C)CCC(C)C(CC(C)CCC(C)C(CC(C)CCC(C)CCC1CCC2CCCCC21)CC1CCC2CCCCC21)CC1CCC2CCCCC21)CC1CCC2CCCCC21',
  activity_type='pIC50',
  ident=[1],
  smiles='NC(=[NH2+])NCCC[C@H](NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCNC(N)=[NH2+])NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCNC(N)=[NH2+])NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@H](CCCNC(N)=[NH2+])NC(=O)[C@H](Cc1c[nH]c2ccccc12)NC(=O)[C@@H]([NH3+])CCCNC(N)=[NH2+])C(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)[O-]',
  ligand={
    z=[126],
    x=[126, 12],
    pos=[126, 3],
  },
  pocket={
    z=[676],
    x=[676, 12],
    pos=[676, 3],
  },
  pocket_residue={ x=[85, 23] },
  complex={
    x=[802, 12],
    z=[802],
    pos=[802, 3],
  },
  (ligand, bond, ligand)={
    edge_index=[2, 270],
    edge_attr=[270, 4],
  },
  (pocket, b

In [10]:
import torch

torch.save(dataset[max_idx], f"{ROOT}/data/probing/targets/outlier_n.pt")